# Day 26 · Break the tools on purpose

**First-principles question:** What does your agent do when a tool times out, rate-limits or returns garbage?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
import random
import time
import pandas as pd
import lab.acme_tools as acme_tools
from lab.acme import load_cases
from lab.agent import run_agent
from lab.evals import score

cases = load_cases()
ORIGINAL = dict(acme_tools.FUNCTIONS)

## 1 · A tool that fails on purpose

In [ ]:
class ToolError(Exception):
    pass

def flaky(fn, mode, rate, rng=random.Random(0)):
    """Wrap a tool so that it fails `rate` of the time in the given mode: "429", "503", "slow" or "garbage"."""
    def wrapped(**args):
        if rng.random() < rate:
            # TODO: "429" and "503": raise ToolError(f"HTTP {mode}")
            #       "slow": time.sleep(5), then return fn(**args)
            #       "garbage": return '{"name": "Pri'   (half a JSON)
            ...
        return fn(**args)
    return wrapped

## 2 · Run every case under each fault

In [ ]:
def outcome(case, out):
    s = score(case, out)
    if s["passed"]:
        return "passed"
    if out.get("needs_human"):
        return "escalated"
    return "invented" if not s["facts_ok"] else "failed"

def run_all(mode, rate=0.3):
    acme_tools.FUNCTIONS.update({k: flaky(v, mode, rate) for k, v in ORIGINAL.items()})
    counts = {"passed": 0, "escalated": 0, "invented": 0, "failed": 0, "crashed": 0}
    try:
        for i, case in enumerate(cases):
            try:
                out, _ = run_agent(case, run=100 + i)
                counts[outcome(case, out)] += 1
            except Exception:
                counts["crashed"] += 1
    finally:
        acme_tools.FUNCTIONS.update(ORIGINAL)
    return counts

before = pd.DataFrame({m: run_all(m) for m in ["429", "503", "slow", "garbage"]}).T
before

## 3 · The fix
In `lab/agent.py`: catch tool errors, retry 429 and 503 with backoff, give the model a clear error, and escalate when a tool keeps failing. Then re-run.

In [ ]:
import importlib, lab.agent
importlib.reload(lab.agent)
from lab.agent import run_agent
after = pd.DataFrame({m: run_all(m) for m in ["429", "503", "slow", "garbage"]}).T
after

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
try:
    flaky(lambda: "ok", "503", 1.0)()
    raise AssertionError("flaky(..., '503', 1.0) should raise")
except ToolError:
    pass
assert flaky(lambda: "ok", "503", 0.0)() == "ok"
assert after["crashed"].sum() == 0, "Step 3: no crashes after the fix"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")